In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

In [3]:
# project folders
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data_raw"
PROCESSED_DIR = PROJECT_DIR / "data_processed"

RAW_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(exist_ok=True)

print("Project folder:", PROJECT_DIR)

Project folder: /Users/aanchal


In [5]:
# Chicago location and date range
LAT = 41.88
LON = -87.63

START_DATE = "2025-01-01"
END_DATE = "2025-06-30"

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "timezone": "America/Chicago",
    "daily": ",".join([
        "weather_code",
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "precipitation_hours"
    ])
}

params

{'latitude': 41.88,
 'longitude': -87.63,
 'start_date': '2025-01-01',
 'end_date': '2025-06-30',
 'timezone': 'America/Chicago',
 'daily': 'weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours'}

In [7]:
# call Open-Meteo API
response = requests.get(BASE_URL, params=params, timeout=60)
response.raise_for_status()

data = response.json()

print("Status code:", response.status_code)
print("Top-level keys:", list(data.keys()))

Status code: 200
Top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']


In [9]:
# save raw JSON
with open(RAW_DIR / "openmeteo_daily_raw.json", "w") as f:
    json.dump(data, f, indent=2)

print("Saved raw JSON to:", RAW_DIR / "openmeteo_daily_raw.json")

Saved raw JSON to: /Users/aanchal/data_raw/openmeteo_daily_raw.json


In [11]:
# convert daily weather block to dataframe
daily_weather = pd.DataFrame(data["daily"])

daily_weather.head()

,time,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours
0,2025-01-01,71,-1.1,0.8,-3.0,0.1,0.0,0.07,1.0
1,2025-01-02,71,-2.4,-0.3,-4.2,0.2,0.0,0.14,2.0
2,2025-01-03,3,-4.9,-3.0,-6.9,0.0,0.0,0.00,0.0
3,2025-01-04,3,-8.0,-5.1,-10.9,0.0,0.0,0.00,0.0
4,2025-01-05,71,-6.6,-2.3,-9.0,0.4,0.1,0.21,3.0


In [13]:
# clean and prepare columns
daily_weather["time"] = pd.to_datetime(daily_weather["time"])
daily_weather = daily_weather.rename(columns={"time": "crash_day"})

# derived flags
daily_weather["wet_day"] = (daily_weather["precipitation_sum"] > 0).astype(int)
daily_weather["snow_day"] = (daily_weather["snowfall_sum"] > 0).astype(int)

daily_weather.head()

,crash_day,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,wet_day,snow_day
0,2025-01-01,71,-1.1,0.8,-3.0,0.1,0.0,0.07,1.0,1,1
1,2025-01-02,71,-2.4,-0.3,-4.2,0.2,0.0,0.14,2.0,1,1
2,2025-01-03,3,-4.9,-3.0,-6.9,0.0,0.0,0.00,0.0,0,0
3,2025-01-04,3,-8.0,-5.1,-10.9,0.0,0.0,0.00,0.0,0,0
4,2025-01-05,71,-6.6,-2.3,-9.0,0.4,0.1,0.21,3.0,1,1


In [15]:
# quick structure check
print("Shape:", daily_weather.shape)
print("\nColumns:")
print(daily_weather.columns.tolist())

daily_weather.describe(include="all")

Shape: (181, 11)

Columns:
['crash_day', 'weather_code', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'precipitation_hours', 'wet_day', 'snow_day']


,crash_day,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,wet_day,snow_day
count,181,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000,181.000000
mean,2025-04-01 00:00:00,33.325967,7.114917,11.412707,3.193923,2.420442,2.156906,0.184475,3.077348,0.519337,0.165746
min,2025-01-01 00:00:00,0.000000,-18.400000,-15.400000,-21.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025-02-15 00:00:00,3.000000,-1.100000,2.700000,-3.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2025-04-01 00:00:00,51.000000,6.600000,11.400000,3.800000,0.100000,0.000000,0.000000,1.000000,1.000000,0.000000
75%,2025-05-16 00:00:00,63.000000,15.000000,20.300000,9.900000,1.500000,0.900000,0.000000,5.000000,1.000000,0.000000
max,2025-06-30 00:00:00,75.000000,30.900000,35.600000,25.900000,41.100000,41.100000,6.160000,20.000000,1.000000,1.000000
std,NaN,30.138547,10.760770,11.523474,10.289505,5.761083,5.698501,0.736238,4.545399,0.501012,0.372884


In [17]:
# save processed weather csv
daily_weather.to_csv(PROCESSED_DIR / "openmeteo_daily_jan_jun_2025.csv", index=False)

print("Saved processed weather CSV to:", PROCESSED_DIR / "openmeteo_daily_jan_jun_2025.csv")

Saved processed weather CSV to: /Users/aanchal/data_processed/openmeteo_daily_jan_jun_2025.csv


In [19]:
# quick report-ready summary
weather_summary = pd.DataFrame({
    "metric": [
        "row_count",
        "column_count",
        "min_day",
        "max_day",
        "wet_day_count",
        "snow_day_count",
        "missing_precipitation_sum",
        "missing_weather_code"
    ],
    "value": [
        daily_weather.shape[0],
        daily_weather.shape[1],
        daily_weather["crash_day"].min(),
        daily_weather["crash_day"].max(),
        int(daily_weather["wet_day"].sum()),
        int(daily_weather["snow_day"].sum()),
        int(daily_weather["precipitation_sum"].isna().sum()),
        int(daily_weather["weather_code"].isna().sum())
    ]
})

weather_summary

,metric,value
0,row_count,181
1,column_count,11
2,min_day,2025-01-01 00:00:00
3,max_day,2025-06-30 00:00:00
4,wet_day_count,94
5,snow_day_count,30
6,missing_precipitation_sum,0
7,missing_weather_code,0


In [21]:
weather_summary.to_csv(PROCESSED_DIR / "openmeteo_weather_summary.csv", index=False)
print("Saved weather summary to:", PROCESSED_DIR / "openmeteo_weather_summary.csv")

Saved weather summary to: /Users/aanchal/data_processed/openmeteo_weather_summary.csv
